In [1]:
from langchain_groq import ChatGroq
from sqlalchemy import create_engine, text, MetaData
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
import os

load_dotenv()

True

In [2]:
model = ChatGroq(model = 'openai/gpt-oss-120b')

In [3]:
#select the database URI from environment variable or use a default SQLite in-memory database
DB_URI = os.getenv("AGENT_DB_URI", "sqlite:///:memory:") 
engine = create_engine(
    DB_URI,
    pool_size=5,          # baseline connections kept open
    max_overflow=10,      # extra connections allowed under burst load (total cap = 15)
    pool_timeout=30,      # seconds to wait for a free connection before erroring
    pool_recycle=1800,    # recycle connections older than 30 min (avoids stale/dropped conns)
    pool_pre_ping=True,   # test connection liveness before handing it out — avoids "server closed the connection" errors
)
metadata = MetaData()

In [ ]:
#begin will start and close a connection after execution 
# for connect you need to commit tooo , but if it is begin : it cna handle on it's own  
with engine.begin() as connection:
    connection.execute(text("CREATE TABLE users  (id INTEGER PRIMARY KEY , name TEXT, role TEXT);"))
    connection.execute(text("INSERT INTO users  (id , name , role) VALUES (1, 'RANA', 'ADMIN'), (2, 'RABI', 'USER'), (3, 'ALI', 'WORKER');"))

In [4]:
from dataclasses import dataclass
from typing import Literal
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from sqlalchemy.exc import OperationalError
import structlog

logger = structlog.get_logger()  # create once at module level, not inside except blocks

# ---------- structured error handling ----------

@dataclass
class ToolError:
    kind: Literal["permission_denied", "syntax_error", "not_found", "timeout", "rejected", "unknown"]
    message: str

    def to_model_text(self) -> str:
        return f"Error ({self.kind}): {self.message}"


def classify_db_error(e: Exception) -> ToolError:
    msg = str(e).lower()
    if "permission denied" in msg:
        return ToolError("permission_denied", str(e))
    if "does not exist" in msg or "no such table" in msg:
        return ToolError("not_found", str(e))
    if "syntax error" in msg:
        return ToolError("syntax_error", str(e))
    if "timeout" in msg:
        return ToolError("timeout", str(e))
    return ToolError("unknown", str(e))


In [5]:
# ---------- retry-wrapped low-level DB call ----------

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=8), retry=retry_if_exception_type(OperationalError))  # only retry transient connection errors
def _run_query(query: str):
    with engine.connect() as connection:
        result = connection.execute(text(query))
        if result.returns_rows:
            rows = result.fetchall()
            column_names = list(result.keys())
            return {"rows": rows, "columns": column_names}
        else:
            connection.commit()
            return {"rowcount": result.rowcount}


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=8), retry=retry_if_exception_type(OperationalError))
def _reflect_schema():
    metadata.reflect(bind=engine)
    return metadata.tables


# ---------- tools ----------

@tool
def get_db_schema():
    """Retrieve this function to get the database schema, such as table and column names.
    Always run this function before running/making any query.
    """
    try:
        tables = _reflect_schema()
        schema_info = []
        for table_name, table in tables.items():
            columns = [f"{col.name} ({col.type})" for col in table.columns]
            schema_info.append(f"Table: {table_name}\n Columns: {', '.join(columns)}")
        return "\n\n".join(schema_info) if schema_info else "The Database is empty."
    except Exception as e:
        err = classify_db_error(e)
        logger.warning("db_tool_error", tool="get_db_schema", kind=err.kind)
        return err.to_model_text()


ALLOWED_STATEMENTS = ("SELECT", "WITH")  # must be a tuple for str.startswith()

def is_read_only(query: str) -> bool:
    stripped = query.strip().upper()
    return stripped.startswith(ALLOWED_STATEMENTS)


@tool
def execute_sql_query(query: str):
    """Execute a read-only SQL query against the database and return the result.
    Only pass valid SELECT/WITH statements to this tool.
    """
    if not is_read_only(query):
        err = ToolError("rejected", "Only SELECT/WITH (read-only) statements are allowed.")
        logger.warning("db_tool_error", tool="execute_sql_query", kind=err.kind, query=query)
        return err.to_model_text()

    try:
        result = _run_query(query)
        if "rows" in result:
            output = [f"Columns : {', '.join(result['columns'])}"]
            for row in result["rows"]:
                output.append(str(dict(zip(result["columns"], row))))
            return "\n".join(output)
        else:
            return f"Query executed successfully, rows affected: {result['rowcount']}"
    except Exception as e:
        err = classify_db_error(e)
        logger.warning("db_tool_error", tool="execute_sql_query", kind=err.kind, query=query)
        return err.to_model_text()


db_tool_kit = [get_db_schema, execute_sql_query]

In [6]:
SYSTEM_PROMPT =( """
        You are a helpful data assistant.
        use your tools to inspect the database and 
        write accurate database queries to answer the user questions/queries """)


In [7]:
agent = create_agent(tools=db_tool_kit , model=model , system_prompt=SYSTEM_PROMPT)

In [8]:
def ask(question: str):
    print(f"\n{'='*70}\nQ: {question}\n{'='*70}")
    for step in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
        config = {'recursion_limit':10}
    ):
        msg = step["messages"][-1]
        if msg.type == "ai" and msg.tool_calls:
            for call in msg.tool_calls:
                print(f"  [reasoning] -> calling tool `{call['name']}`"
                      f" with args {call['args']}")
        elif msg.type == "tool":
            print(f"  [observation] {msg.name} returned: {msg.content[:200]}")
        elif msg.type == "ai" and msg.content:
            print(f"  [final answer] {msg.content}")
 

if __name__ == "__main__":
    if not os.environ.get("GROQ_API_KEY"):
        print("Set GROQ_API_KEY before running this file.")
    else:
        ask("What is the schema of the data you have access too")
        ask("How many users are in the 'users' table ")


Q: What is the schema of the data you have access too
  [reasoning] -> calling tool `get_db_schema` with args {}
  [observation] get_db_schema returned: Table: checkpoint_migrations
 Columns: v (INTEGER)

Table: checkpoints
 Columns: thread_id (TEXT), checkpoint_ns (TEXT), checkpoint_id (TEXT), parent_checkpoint_id (TEXT), type (TEXT), checkpoint (JSO
  [final answer] Here’s the full schema of the SQLite‑style database I can query:

| Table | Columns (type) |
|-------|----------------|
| **checkpoint_migrations** | `v` (INTEGER) |
| **checkpoints** | `thread_id` (TEXT), `checkpoint_ns` (TEXT), `checkpoint_id` (TEXT), `parent_checkpoint_id` (TEXT), `type` (TEXT), `checkpoint` (JSONB), `metadata` (JSONB) |
| **checkpoint_blobs** | `thread_id` (TEXT), `checkpoint_ns` (TEXT), `channel` (TEXT), `version` (TEXT), `type` (TEXT), `blob` (BYTEA) |
| **checkpoint_writes** | `thread_id` (TEXT), `checkpoint_ns` (TEXT), `checkpoint_id` (TEXT), `task_id` (TEXT), `idx` (INTEGER), `channel` (TEXT), `